In [2]:
# ==========================================
# YÊU CẦU 1: TF-IDF cho review_comment_message
# ==========================================

import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# tải stopwords tiếng Bồ Đào Nha
nltk.download('stopwords')
stop_words_pt = stopwords.words('portuguese')

# đọc dữ liệu
df = pd.read_parquet('..\Data\Raw\master_dataset.parquet')

# xử lý null (giữ lại toàn bộ dòng)
df_reviews = df.copy()
df_reviews['review_comment_message'] = df_reviews['review_comment_message'].fillna("")

# TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    stop_words=stop_words_pt,
    max_features=500,
    lowercase=True,
    min_df=5,
    token_pattern=r'(?u)\b[a-zA-ZÀ-ÿ]{2,}\b'
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df_reviews['review_comment_message'])

print("Kích thước ma trận TF-IDF:", tfidf_matrix.shape)
print("Top 20 từ khóa:", tfidf_vectorizer.get_feature_names_out()[:20])

# lưu model TF-IDF
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.joblib')
print("✅ Đã lưu tfidf_vectorizer.joblib")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Kích thước ma trận TF-IDF: (113425, 500)
Top 20 từ khóa: ['abri' 'abrir' 'absurdo' 'acabamento' 'achei' 'acho' 'aconteceu' 'acordo'
 'acredito' 'adorei' 'agilidade' 'agora' 'agradeço' 'aguardando' 'aguardo'
 'ainda' 'algum' 'alguns' 'além' 'amei']
✅ Đã lưu tfidf_vectorizer.joblib


In [4]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

df = pd.read_parquet('..\Data\Raw\master_dataset.parquet')

cat_col = 'product_category_name_english' if 'product_category_name_english' in df.columns else 'product_category_name'
df_items = df.dropna(subset=['order_id', cat_col])

transactions = (
    df_items.groupby('order_id')[cat_col]
    .apply(lambda x: list(set(x)))
    .tolist()
)

# lọc đơn có >=2 sản phẩm
transactions = [t for t in transactions if len(t) >= 2]

print("Đơn hàng có >=2 sản phẩm:", len(transactions))

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_trans = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = fpgrowth(df_trans, min_support=0.0001, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.01)

print("Frequent itemsets:", frequent_itemsets.shape)
print("Rules:", rules.shape)

top_rules = rules.sort_values(by=['lift','confidence'], ascending=False).head(10)

# đổi sang string để CSV dễ đọc
top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: ','.join(list(x)))
top_rules['consequents'] = top_rules['consequents'].apply(lambda x: ','.join(list(x)))

print("\n🏆 Top 10 rules:")
print(top_rules[['antecedents','consequents','support','confidence','lift']])

top_rules.to_csv('fp_growth_rules.csv', index=False)
print("✅ Đã lưu fp_growth_rules.csv")

<>:5: DeprecationWarning: invalid escape sequence '\D'
<>:5: DeprecationWarning: invalid escape sequence '\D'
C:\Users\Admin\AppData\Local\Temp\ipykernel_2164\409481460.py:5: DeprecationWarning: invalid escape sequence '\D'
  df = pd.read_parquet('..\Data\Raw\master_dataset.parquet')


Đơn hàng có >=2 sản phẩm: 726
Frequent itemsets: (317, 2)
Rules: (519, 14)

🏆 Top 10 rules:
                       antecedents                    consequents   support  \
490         fashio_female_clothing                  fashion_sport  0.001377   
491                  fashion_sport         fashio_female_clothing  0.001377   
177  auto,fashion_bags_accessories            musical_instruments  0.001377   
178            musical_instruments  auto,fashion_bags_accessories  0.001377   
516                          music                books_technical  0.001377   
515                books_technical                          music  0.001377   
175       musical_instruments,auto       fashion_bags_accessories  0.001377   
518      fashion_childrens_clothes       fashion_bags_accessories  0.002755   
517       fashion_bags_accessories      fashion_childrens_clothes  0.002755   
180       fashion_bags_accessories       musical_instruments,auto  0.001377   

     confidence        lift  
490    1

In [5]:
print(top_rules[['antecedents','consequents','support','confidence','lift']].head(10).to_string(index=False))

                  antecedents                   consequents  support  confidence       lift
       fashio_female_clothing                 fashion_sport 0.001377    1.000000 363.000000
                fashion_sport        fashio_female_clothing 0.001377    0.500000 363.000000
auto,fashion_bags_accessories           musical_instruments 0.001377    1.000000 103.714286
          musical_instruments auto,fashion_bags_accessories 0.001377    0.142857 103.714286
                        music               books_technical 0.001377    0.500000  90.750000
              books_technical                         music 0.001377    0.250000  90.750000
     musical_instruments,auto      fashion_bags_accessories 0.001377    1.000000  66.000000
    fashion_childrens_clothes      fashion_bags_accessories 0.002755    1.000000  66.000000
     fashion_bags_accessories     fashion_childrens_clothes 0.002755    0.181818  66.000000
     fashion_bags_accessories      musical_instruments,auto 0.001377    0.090909